# PCA from the covariance matrix

The lesson's class takes the SVD of the centered data. This one starts where the
derivation does, from the sample covariance matrix and `np.linalg.eigh`, and adds three
things a usable PCA needs: a fixed sign for every component, so that two runs agree, a
number of components given as a fraction of the variance, and whitened scores that can be
turned back into rows. The data is a week of readings from six strain gauges on a bridge.

Your code may use NumPy, including `np.linalg.eigh`, but not `np.linalg.svd` or
scikit-learn.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO`
and the `raise NotImplementedError` with your code, then run the check cell under it. The
check prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 300 readings from six strain gauges. Three load patterns drive the gauges, plus noise,
# around very different resting levels.
rng = np.random.default_rng(11)
n = 300
patterns = np.array([[3.0, 2.5, 2.0, 1.0, 0.5, 0.2],
                     [0.0, -1.0, 1.5, 2.0, -0.5, 1.0],
                     [0.5, 0.0, -0.5, 0.5, 1.5, -1.0]])
X = rng.normal(size=(n, 3)) @ patterns + rng.normal(scale=0.3, size=(n, 6)) + np.array([20, 35, 50, 12, 8, 41])
print("X", X.shape)

## Part 1 (coding task): Fit

Write `pca_fit(X, n_components)` for a data matrix `X` of shape `(n, d)`. It returns four
arrays:

- `mean`, shape `(d,)`: the column means.
- `components`, shape `(k, d)`: one unit-length principal component per row, in order of
  decreasing variance, each flipped if needed so that its entry of largest absolute value
  is positive.
- `variances`, shape `(k,)`: the variance of the data along each kept component, an
  eigenvalue of the sample covariance matrix (dividing by $n - 1$).
- `ratios`, shape `(k,)`: each kept variance divided by the total variance of the data.

`n_components` is either an integer $k$ with $1 \le k \le d$, or a float $f$ with
$0 < f < 1$, which keeps the smallest $k$ whose cumulative ratio is at least $f$.

In [ ]:
def pca_fit(X, n_components):
    X = np.asarray(X, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
def _banned(*args, **kwargs):
    raise AssertionError("np.linalg.svd is not allowed in your code: build the components from the covariance matrix")

def _call(f, *args, **kwargs):
    saved = np.linalg.svd
    np.linalg.svd = _banned
    try:
        return f(*args, **kwargs)
    finally:
        np.linalg.svd = saved

def _reference(X):
    mu = X.mean(axis=0)
    _, s, vt = np.linalg.svd(X - mu, full_matrices=False)
    var = s ** 2 / (len(X) - 1)
    vt = vt * np.sign(vt[np.arange(len(vt)), np.argmax(np.abs(vt), axis=1)])[:, None]
    return mu, vt, var, var / var.sum()

r_mean, r_comp, r_var, r_ratio = _reference(X)
mean, comp, var, ratio = _call(pca_fit, X, 2)
assert np.shape(mean) == (6,) and np.shape(comp) == (2, 6) and np.shape(var) == (2,) and np.shape(ratio) == (2,), \
    f"shapes {np.shape(mean)}, {np.shape(comp)}, {np.shape(var)}, {np.shape(ratio)}: expected (6,), (2, 6), (2,), (2,)"
assert np.allclose(mean, r_mean), "mean: expected the column means of X"
if np.allclose(var, r_var[:2] * (n - 1) / n):
    raise AssertionError("variances: the sample covariance divides by n - 1, not n")
assert np.allclose(var, r_var[:2]), \
    f"variances {np.round(var, 3)}: expected the two largest eigenvalues of the sample covariance, largest first, {np.round(r_var[:2], 3)}"
if np.allclose(np.abs(comp), np.abs(r_comp[:2])) and not np.allclose(comp, r_comp[:2]):
    raise AssertionError("components: right directions, wrong signs. Flip each one so that its entry of largest absolute value is positive")
assert np.allclose(comp, r_comp[:2]), "components: expected the eigenvectors of the two largest eigenvalues, one per row"
if np.isclose(np.sum(ratio), 1.0):
    raise AssertionError("ratios add up to 1 over the kept components: divide by the total variance of all d directions")
assert np.allclose(ratio, r_ratio[:2]), f"ratios {np.round(ratio, 4)}: expected {np.round(r_ratio[:2], 4)}"

cum = np.cumsum(r_ratio)
for f, k in [(cum[2] - 1e-9, 3), (cum[2] + 1e-9, 4), (cum[0] / 2, 1), (float(cum[1] + 1e-6), 3)]:
    got = _call(pca_fit, X, float(f))[1]
    assert len(got) == k, f"n_components={f:.6f}: kept {len(got)} components, expected {k}, the smallest number whose cumulative ratio reaches the fraction"
assert len(_call(pca_fit, X, 6)[1]) == 6, "n_components=6 should keep all six components"

shifted = X + 1000.0
assert np.allclose(_call(pca_fit, shifted, 3)[1], r_comp[:3]), "adding 1000 to every reading should not change the components"
print("cumulative ratios", np.round(cum, 4))
print("Part 1 passed")

## Part 2 (coding task): Scores and the way back

Write two functions that use the arrays from `pca_fit`.

- `pca_transform(X, mean, components, variances, whiten=False)` returns the scores of the
  rows of `X`, shape `(len(X), k)`. With `whiten=True`, each column of scores is scaled so
  that, on the data the PCA was fitted on, it has sample variance 1.
- `pca_inverse(Z, mean, components, variances, whiten=False)` takes scores made with the
  same `whiten` setting and returns rows in the original coordinates.

The check also tests the reconstruction error of two components against the variance they
leave out.

In [ ]:
def pca_transform(X, mean, components, variances, whiten=False):
    # TODO
    raise NotImplementedError


def pca_inverse(Z, mean, components, variances, whiten=False):
    # TODO
    raise NotImplementedError

In [ ]:
def _banned(*args, **kwargs):
    raise AssertionError("np.linalg.svd is not allowed in your code: build the components from the covariance matrix")

def _call(f, *args, **kwargs):
    saved = np.linalg.svd
    np.linalg.svd = _banned
    try:
        return f(*args, **kwargs)
    finally:
        np.linalg.svd = saved

def _reference(X):
    mu = X.mean(axis=0)
    _, s, vt = np.linalg.svd(X - mu, full_matrices=False)
    var = s ** 2 / (len(X) - 1)
    vt = vt * np.sign(vt[np.arange(len(vt)), np.argmax(np.abs(vt), axis=1)])[:, None]
    return mu, vt, var, var / var.sum()

full = _call(pca_fit, X, 6)[:3]    # mean, components, variances
two = _call(pca_fit, X, 2)[:3]

Z = pca_transform(X, *two)
assert np.shape(Z) == (n, 2), f"scores shape {np.shape(Z)}: expected ({n}, 2)"
assert np.allclose(np.cov(Z.T), np.diag(two[2])), "unwhitened scores should have sample covariance diag(variances)"
W = pca_transform(X, *two, whiten=True)
if np.allclose(np.cov(W.T), np.diag(1 / two[2])):
    raise AssertionError("whitened scores have variance 1 / variance: divide by the standard deviation, not the variance")
assert np.allclose(np.cov(W.T), np.eye(2)), "whitened scores should have sample covariance equal to the identity"

few = X[:3]
assert np.allclose(pca_transform(few, *two), (few - two[0]) @ two[1].T), \
    "scores of three rows: center with the mean fitted on the training data, not the mean of the rows passed in"

for whiten in (False, True):
    back = pca_inverse(pca_transform(X, *full, whiten=whiten), *full, whiten=whiten)
    assert np.allclose(back, X), f"whiten={whiten}: with all six components, inverse(transform(X)) should return X"

X_hat = pca_inverse(Z, *two)
error = ((X - X_hat) ** 2).sum(axis=1).mean()
left_out = full[2][2:].sum() * (n - 1) / n
assert np.isclose(error, left_out), f"mean squared reconstruction error {error:.4f}, expected {left_out:.4f}"
print(f"two components: mean squared reconstruction error {error:.4f}, (n - 1)/n times the left-out variance {left_out:.4f}")
print("Part 2 passed")

## Solution

### Part 1: Fit

In [ ]:
def pca_fit(X, n_components):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    Xc = X - mean
    cov = Xc.T @ Xc / (len(X) - 1)
    vals, vecs = np.linalg.eigh(cov)                 # ascending order
    order = np.argsort(vals)[::-1]
    vals, vecs = np.clip(vals[order], 0, None), vecs[:, order]
    components = vecs.T                              # one component per row
    biggest = np.argmax(np.abs(components), axis=1)
    components = components * np.sign(components[np.arange(len(components)), biggest])[:, None]
    ratios = vals / vals.sum()
    if isinstance(n_components, (float, np.floating)) and 0 < n_components < 1:
        k = int(np.searchsorted(np.cumsum(ratios), n_components)) + 1
    else:
        k = int(n_components)
    return mean, components[:k], vals[:k], ratios[:k]

**Why it works.** The variance along a unit vector $\mathbf{w}$ is $\mathbf{w}^\top \Sigma\,\mathbf{w}$, largest
at the top eigenvector, so the components are the eigenvectors of the covariance sorted by
eigenvalue. `eigh` returns them in ascending order, which is why the code reverses them,
and in columns, which is why it transposes. The eigenvalues add up to the trace, the total
variance, so each ratio divides by the sum over all $d$ of them. An eigenvector is defined
only up to sign, and fixing the sign of the largest entry makes the output the same on
every machine. For a fraction, `searchsorted` returns the number of cumulative ratios below
it, so one more component is the first to reach it. On the gauges, the first three
components explain about 0.99 of the variance: three load patterns drive six gauges.

### Part 2: Scores and the way back

In [ ]:
def pca_transform(X, mean, components, variances, whiten=False):
    Z = (np.asarray(X, dtype=float) - mean) @ components.T
    return Z / np.sqrt(variances) if whiten else Z


def pca_inverse(Z, mean, components, variances, whiten=False):
    Z = np.asarray(Z, dtype=float)
    if whiten:
        Z = Z * np.sqrt(variances)
    return Z @ components + mean

**Why it works.** The scores are coordinates in the basis of the components, measured from the fitted mean,
so a new batch of rows is centered with the training mean and not its own. The scores
along component $j$ have variance $\lambda_j$, so dividing by $\sqrt{\lambda_j}$, the
standard deviation, gives variance 1, and the inverse multiplies it back before mapping
the scores to rows with `Z @ components` and adding the mean. With all six components that
map is a rotation and the round trip is exact. With two, the mean squared error per row is
the left-out variance times $(n - 1)/n$, the factor between a sample variance and a plain
mean of squares.